# Logistic Regression Classification with Supermarket Dataset

I will be using the Logistic Regression algorithm to predict whether a supermarket transaction will result in a repeated purchase (`repeated_purchase`). The dataset contains transaction records and features covering customer behavior, pricing, promotions, and shopping habits.

We will be importing the pandas library to read and analyze the file, along with the scikit-learn module for Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

### Read the csv (comma separated file) using pandas

In [ ]:
supermarketData = pd.read_csv("supermarket.csv")
supermarketData.head()

,transaction_id,transaction_line_id,basket_id,customer_id,store_id,product_id,employee_id,supplier_id,customer_age,customer_gender,...,units_sold,sales_revenue,profit_per_unit,profit_margin,sales_rank,category_sales_rank,product_popularity,product_growth_rate,product_return_rate,product_stockout_rate
0,T00000001,TL00000001,B00044698,C00106069,S00000108,P00003814,E00000421,SUP00000007,60,Female,...,10,100.60,4.65,0.462,883,10,5.6,0.064,0.0580,0.0293
1,T00000002,TL00000002,B00013390,C00027234,S00000154,P00003904,E00000310,SUP00000035,54,Male,...,6,96.90,8.66,0.536,599,29,6.0,-0.025,0.0071,0.0028
2,T00000003,TL00000003,B00014357,C00019781,S00000136,P00000650,E00000344,SUP00000022,52,Male,...,6,132.18,7.90,0.359,389,49,5.2,-0.038,0.0029,0.0553
3,T00000004,TL00000004,B00046655,C00018191,S00000169,P00000966,E00000168,SUP00000005,40,Male,...,12,54.36,1.97,0.435,111,70,3.8,0.189,0.0146,0.0526
4,T00000005,TL00000005,B00046461,C00031493,S00000179,P00000584,E00000455,SUP00000014,44,Female,...,5,21.75,0.04,0.009,794,7,5.8,0.116,0.0479,0.1025


### Check Dataset Columns and Shape
Let's inspect our column names and overall dataset dimensions. The target variable we want to predict is `repeated_purchase`.
### We can check the different column names in the dataset as follows:

In [38]:

print("Dataset shape (rows, columns):", supermarketData.shape)
supermarketData.columns

Dataset shape (rows, columns): (120000, 323)


Index(['transaction_id', 'transaction_line_id', 'basket_id', 'customer_id',
       'store_id', 'product_id', 'employee_id', 'supplier_id', 'customer_age',
       'customer_gender',
       ...
       'units_sold', 'sales_revenue', 'profit_per_unit', 'profit_margin',
       'sales_rank', 'category_sales_rank', 'product_popularity',
       'product_growth_rate', 'product_return_rate', 'product_stockout_rate'],
      dtype='object', length=323)

### Checking Missing Values
Instead of dropping rows (which would delete the entire dataset), we will inspect null values and let our pipeline handle imputation automatically.

In [39]:
print(f"Total null values in dataset: {supermarketData.isnull().sum().sum()}")

Total null values in dataset: 1122101


### Shuffling the Dataset
Shuffling breaks any pre-existing chronological or class-based ordering in the dataset before we split it.

In [40]:
df_randomized = supermarketData.sample(frac=1, random_state=42).reset_index(drop=True)
df_randomized.head()

,transaction_id,transaction_line_id,basket_id,customer_id,store_id,product_id,employee_id,supplier_id,customer_age,customer_gender,...,units_sold,sales_revenue,profit_per_unit,profit_margin,sales_rank,category_sales_rank,product_popularity,product_growth_rate,product_return_rate,product_stockout_rate
0,T00071788,TL00071788,B00010130,C00043884,S00000089,P00004044,E00000455,SUP00000050,59,Male,...,4,26.24,2.36,0.360,633,75,5.0,0.039,0.0578,0.0756
1,T00067219,TL00067219,B00006989,C00076035,S00000185,P00002724,E00000205,SUP00000033,28,Male,...,6,26.28,-0.17,-0.039,228,47,2.7,0.037,0.0298,0.0310
2,T00054067,TL00054067,B00000699,C00020444,S00000153,P00003449,E00000136,SUP00000033,24,Male,...,8,123.68,1.10,0.071,182,93,4.1,-0.067,0.0495,0.0173
3,T00007169,TL00007169,B00036981,C00077518,S00000075,P00001380,E00000011,SUP00000044,70,Female,...,6,32.16,2.06,0.384,637,74,4.4,-0.417,0.0222,0.0565
4,T00029619,TL00029619,B00049286,C00067742,S00000034,P00001764,E00000295,SUP00000014,51,Male,...,11,29.70,0.69,0.256,900,20,4.1,-0.173,0.0406,0.0309


### Separating Features (X) and Target (y)
We separate our target label (`repeated_purchase`) from all other feature columns.

In [ ]:
target_col = 'repeated_purchase'
for col in supermarketData.columns:
    if 'repeat' in col.lower() or 'purchase' in col.lower():
        target_col = col
        break

drop_cols = [target_col, 'transaction_id', 'transaction_line_id', 'basket_id', 
             'customer_id', 'store_id', 'product_id', 'employee_id', 'supplier_id',
             'transaction_date', 'transaction_time', 'transaction_datetime']

existing_drop_cols = [col for col in drop_cols if col in supermarketData.columns]

X = supermarketData.drop(columns=existing_drop_cols)
y = supermarketData[target_col]

print("Features (X) shape after dropping IDs:", X.shape)
print("Target (y) shape:", y.shape)

Features (X) shape after dropping IDs: (120000, 311)
Target (y) shape: (120000,)


### Splitting Data into Training and Test Sets
We reserve 20% of the data for testing and 80% for training.

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=12, shuffle=True
)

print("Training set shape (X_train):", X_train.shape)
print("Test set shape (X_test):", X_test.shape)

Training set shape (X_train): (96000, 311)
Test set shape (X_test): (24000, 311)


## Logistic Regression Model Pipeline & Training
We set up a preprocessing pipeline using `SimpleImputer` (to fill missing numeric/categorical values) and `StandardScaler` / `OneHotEncoder`, followed by our `LogisticRegression` classifier.

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

modelPrediction = model.predict(X_test)
print("Model training and prediction completed successfully!")

/opt/miniconda3/envs/AC312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model training and prediction completed successfully!


## Model Evaluation and Accuracy
We calculate model **accuracy** as the ratio of correct predictions to the total number of predictions, and display the classification report.

$$Accuracy = \frac{\text{Number of correct predictions}}{\text{Total number of predictions}}$$

In [44]:
correct_preds = np.sum(y_test == modelPrediction)
accuracy = correct_preds / modelPrediction.size

print(f"Number of correct predictions: {correct_preds}")
print(f"Total number of predictions: {modelPrediction.size}")
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, modelPrediction))

Number of correct predictions: 8502
Total number of predictions: 24000
Accuracy: 0.3543

Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.76      0.64      8696
           1       0.21      0.17      0.18      4479
           2       0.16      0.11      0.13      3387
           3       0.15      0.11      0.12      2451
           4       0.13      0.10      0.12      1868
           5       0.11      0.09      0.10      1345
           6       0.10      0.08      0.09       932
           7       0.12      0.08      0.10       568
           8       0.11      0.10      0.10       274

    accuracy                           0.35     24000
   macro avg       0.18      0.18      0.18     24000
weighted avg       0.30      0.35      0.32     24000

